# Prompt 09B — Score the **original** HaluEval answers

**Goal:** Compute question-conditioned answer-token NLL with `distilgpt2`, then save mean NLL, 90th-percentile NLL, and answer-token length. **No final test metrics in this notebook.**

This scores *the labelled ChatGPT answer itself*, not a newly generated answer. DistilGPT-2 is a **surrogate scorer**; its probabilities are not the original ChatGPT model's probabilities.

**Prerequisite:** Upload `AITR_Prompt09_data.zip` downloaded in Prompt 09A. Keep the train/validation/test splits unchanged. GPU runtime recommended.


In [ ]:
import os, sys, json, zipfile, hashlib, time
from pathlib import Path
import numpy as np, pandas as pd, torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from google.colab import files
print('=== 09B ENVIRONMENT ===')
print('torch:',torch.__version__, 'CUDA:',torch.cuda.is_available())
print('GPU:',torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')
print('transformers available')


In [ ]:
print('Upload the ZIP generated in Prompt 09A (AITR_Prompt09_data.zip).')
uploaded=files.upload()
zipnames=[x for x in uploaded if x.lower().endswith('.zip')]
if len(zipnames)!=1: raise ValueError('Upload exactly one ZIP from Prompt 09A, then rerun this cell.')
base=Path('/content/aitr_prompt09b'); base.mkdir(exist_ok=True)
with zipfile.ZipFile(zipnames[0]) as z:
    names=z.namelist(); print('ZIP entries:',names)
    for key in ['train.csv','validation.csv','test.csv','dataset_validation.json']:
        matches=[x for x in names if x.endswith('/'+key) or x==key]
        if len(matches)!=1: raise ValueError(f'Expected exactly one {key} in ZIP; got {matches}')
        target=base/key; target.write_bytes(z.read(matches[0]))
report=json.loads((base/'dataset_validation.json').read_text())
print('Original validation report:',report)
parts={k:pd.read_csv(base/f'{k}.csv') for k in ['train','validation','test']}
for name,df in parts.items():
    needed={'user_query','chatgpt_response','label'}
    if not needed.issubset(df.columns):raise ValueError(f'{name}: missing {needed-set(df.columns)}')
    if df[list(needed)].isna().any().any():raise ValueError(f'{name}: null in required columns')
    if not set(df.label.unique()).issubset({0,1}):raise ValueError(f'{name}: bad labels')
    print(name,df.shape,'labels',df.label.value_counts().to_dict())
assert sum(len(x) for x in parts.values())==600, 'Unexpected sample size'


In [ ]:
MODEL_ID='distilgpt2'
tok=AutoTokenizer.from_pretrained(MODEL_ID, use_fast=True)
model=AutoModelForCausalLM.from_pretrained(MODEL_ID)
device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device).eval()
assert tok.is_fast, 'Fast tokenizer required for token boundary offsets'
max_ctx=int(model.config.n_positions)
print('Model:',MODEL_ID,'Device:',device,'Context tokens:',max_ctx)


In [ ]:
# The scoring protocol is fixed: P(answer tokens | "Question: ...\nAnswer: ").
# Tokens that cross the prompt/answer boundary are EXCLUDED, and counted for auditing.
# For long sequences, drop the earliest context tokens, never answer tokens.
def score_one(question, answer):
    prompt=f'Question: {question}\nAnswer: '
    full=prompt+answer
    enc=tok(full,add_special_tokens=False,return_offsets_mapping=True,truncation=False)
    ids=enc['input_ids']; offsets=enc['offset_mapping']; boundary=len(prompt)
    answer_positions=[i for i,(start,end) in enumerate(offsets) if start>=boundary and end>start]
    crossing=sum(1 for start,end in offsets if start<boundary<end)
    if not answer_positions:return {'status':'no_answer_tokens','boundary_crossing_tokens':crossing}
    # Need a previous token to predict the first scored token.
    if len(answer_positions)>=max_ctx:return {'status':'answer_too_long','boundary_crossing_tokens':crossing}
    trim=max(0,len(ids)-max_ctx)
    ids=ids[trim:]
    answer_positions=[i-trim for i in answer_positions if i>=trim]
    answer_positions=[i for i in answer_positions if i>0]
    if not answer_positions:return {'status':'no_predictable_answer_tokens','boundary_crossing_tokens':crossing}
    inp=torch.tensor([ids],dtype=torch.long,device=device)
    with torch.inference_mode():
        logits=model(input_ids=inp).logits[0,:-1,:]
        targets=inp[0,1:]
        # Cross entropy yields per-token negative log-probabilities without retaining huge softmax tensors.
        losses=torch.nn.functional.cross_entropy(logits.float(),targets,reduction='none')
        indices=torch.tensor([i-1 for i in answer_positions],device=device)
        values=losses.index_select(0,indices).cpu().numpy()
    return {'status':'ok','mean_nll':float(values.mean()),'p90_nll':float(np.percentile(values,90)),
            'answer_token_length':int(len(values)), 'context_tokens_dropped':int(trim),
            'boundary_crossing_tokens':int(crossing)}

# Smoke test: both scores must be finite and answer token count > 0.
test=score_one('What is the capital of France?','Paris is the capital of France.')
print('=== SCORING SMOKE TEST ===',test)
assert test['status']=='ok' and np.isfinite(test['mean_nll']) and test['answer_token_length']>0


In [ ]:
# Score each split; writes checkpoints after every 25 rows so progress is not lost.
# Running 600 responses may take several minutes, especially for long answers.
SAVE_EVERY=25
all_counts={}
for split,frame in parts.items():
    output=base/f'{split}_scores.csv'
    rows=[]; start=time.time()
    print(f'=== SCORING {split.upper()} ({len(frame)} rows) ===',flush=True)
    for idx,rec in frame.reset_index(drop=True).iterrows():
        try: score=score_one(str(rec['user_query']),str(rec['chatgpt_response']))
        except torch.cuda.OutOfMemoryError:
            torch.cuda.empty_cache()
            score={'status':'gpu_out_of_memory'}
        row={'row_index':int(idx),'ID':str(rec['ID']) if 'ID' in frame.columns else str(idx),
             'label':int(rec['label']),**score}
        rows.append(row)
        if (idx+1)%SAVE_EVERY==0 or idx+1==len(frame):
            pd.DataFrame(rows).to_csv(output,index=False)
            print(f'{split}: {idx+1}/{len(frame)} | elapsed {time.time()-start:.1f}s',flush=True)
    result=pd.DataFrame(rows)
    all_counts[split]=result.status.value_counts().to_dict()
    print('Status:',all_counts[split])
print('=== SCORING SUMMARY ===',all_counts)


In [ ]:
# Check and package reproducible scoring outputs.
summary={'model_id':MODEL_ID,'context_length':max_ctx,'device':str(device),
 'seed':report.get('seed'),'source_dataset_sha256':report.get('sha256'),
 'score_definition':'answer-token NLL conditional on Question: {user_query}\nAnswer: ',
 'methods':['mean_nll','p90_nll','answer_token_length'],
 'boundary_policy':'exclude tokens crossing prompt-answer boundary',
 'context_policy':'left-truncate context only; skip if answer exceeds context',
 'split_status_counts':all_counts,'note':'Surrogate DistilGPT-2 scores; not original ChatGPT probabilities'}
(base/'scoring_manifest.json').write_text(json.dumps(summary,indent=2))
for split in parts:
    d=pd.read_csv(base/f'{split}_scores.csv')
    assert len(d)==len(parts[split])
    print(split,'scored',int((d.status=='ok').sum()),'/',len(d),'boundary crossing total',int(d.boundary_crossing_tokens.fillna(0).sum()))
zip_path=Path('/content/AITR_Prompt09B_scores.zip')
with zipfile.ZipFile(zip_path,'w',compression=zipfile.ZIP_DEFLATED) as z:
    for file in sorted(base.iterdir()):z.write(file,arcname=file.name)
print('=== OUTPUT READY ===',zip_path,'bytes',zip_path.stat().st_size)
files.download(str(zip_path))
print('STOP: share SCORING SMOKE TEST, SCORING SUMMARY, and scored counts with ChatGPT. Keep the ZIP.')
